![JohnSnowLabs](https://nlp.johnsnowlabs.com/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp-workshop/blob/master/tutorials/Certification_Trainings/Healthcare/51.DecisionModel.ipynb)

If you are using the `johnsnowlabs` library, please use this

---

[39.0.DecisionModel](https://github.com/JohnSnowLabs/spark-nlp-workshop/blob/master/healthcare-nlp/39.0.DecisionModel.ipynb) notebook.

## Healthcare NLP for Data Scientists Course

If you are not familiar with the components in this notebook, you can check [Healthcare NLP for Data Scientists Udemy Course](https://www.udemy.com/course/healthcare-nlp-for-data-scientists/) and the [MOOC Notebooks](https://github.com/JohnSnowLabs/spark-nlp-workshop/tree/master/Spark_NLP_Udemy_MOOC/Healthcare_NLP) for each components.

#   **📜 DecisionModel**

`DecisionModel` is a licensed Healthcare NLP annotator (`DOCUMENT → CATEGORY`), new in Healthcare NLP 7.0.1, that returns **one decision per document**. In this release it runs the **PHI Guard** model family (`getModelType()` returns `phi-guard`).

PHI Guard answers the question every de-identification project has to answer at scale: *is any PHI left in the de-identified output, and how serious is it?* It can also tell which original notes contain any PHI at all, before de-identification. Instead of reviewing samples by hand, you can score every de-identified document, release the ones that pass, and send the rest to review.

`DecisionModel` runs natively in Spark NLP pipelines, with batching, and is saved and loaded with the pipeline like any other stage. All seven PHI Guard models run on CPU with ONNX and need about 1 GB of RAM. For PHI leak detection, PHI Guard v0.7 reaches a PR-AUC of **0.986** on de-identified clinical notes at **6.4 ms** per note on one CPU thread.

**Modes: the four questions PHI Guard answers**

| Mode | Question it answers | Score it on | Labels | Score in metadata |
| --- | --- | --- | --- | --- |
| `leak` | Is any PHI left after de-identification? | the de-identified document | `PHI_LEAK`, `NO_PHI_LEAK` | `phi_leak_prob` |
| `harmful` | Is a remaining leak a harmful one? | the de-identified document | `HARMFUL_PHI_LEAK`, `NO_HARMFUL_PHI_LEAK` | `harmful_leak_prob` |
| `richness` | How much PHI does the note expose? | the de-identified document (or the original note) | `NONE`, `TRACE`, `LOW`, `MODERATE`, `HIGH`, `VERY_HIGH`, `SATURATED` (levels 0 to 6) | `level`, `score_1` … `score_6` |
| `raw` | Does the original note contain PHI? | the original note, before de-identification | `PHI_PRESENT`, `NO_PHI_PRESENT` | `phi_present_prob` |

- **Single-task models (v0.6)** score one mode and need no `setMode()`.
- **Multi-task models (v0.7)** hold all four modes, and one stage scores one of them. Select it with `setMode()` before scoring, and `copy()` the stage for every mode you need, each with its own output column. `getClasses()` returns the labels of the selected mode and fails until a mode is set.
- In the model names, `3m`, `870k` and `850k` give the rounded number of parameters. The `fast` variants are about 4 to 5 times faster per note and a little less accurate.
- A harmful leak is always a leak too: `harmful_leak_prob` never exceeds the leak probability of the same document. In `richness` mode, `score_1` … `score_6` are the probabilities that the note reaches at least each level, and `level` is the number of them at 0.5 or higher.

**Typical uses**

- **Quality gate after de-identification:** add `leak` (and `harmful`) at the end of your de-identification pipeline, release the documents that pass, and send the rest to human review.
- **Monitoring:** track `richness` across batches to see how much PHI is left in your de-identified output, and whether a change in the source data or the pipeline moved it.
- **Triage before de-identification:** run `raw` on the original notes to find the ones that need de-identification at all, or to prioritize them.

**Good to know**

- **Long documents:** PHI Guard reads a document in overlapping windows and combines them into one decision; `n_windows` in the metadata gives the number of windows read. Documents up to 2,000,000 UTF-8 bytes are scored.
- **Check `status` in the metadata:** `scored` means a model decision. `empty` means an empty document, which gets the negative label without running the model (`UNKNOWN` in `richness` mode). `overlength` means a document over the size limit, which gets `REVIEW_REQUIRED` and no probability. Filter on `status == "scored"` before releasing documents on a negative label.
- **Choose your own cutoff:** the labels use a cutoff of 0.5. If your release process must catch nearly every leak, apply a lower cutoff to the probability in the metadata (for example `phi_leak_prob`), chosen on validation data of your own.
- **Sentences or chunks:** the input can also be sentences (`SentenceDetector`) or chunks (`InternalDocumentSplitter`); each one then gets its own decision.
- **A quality signal, not a certificate:** PHI Guard does not find or remove PHI, and a negative label does not guarantee that a document is safe to release. Use it alongside your review process, not instead of it.


## **🎬 Colab Setup**

In [ ]:
import json
import os

from google.colab import files

license_keys = files.upload()

with open(list(license_keys.keys())[0]) as f:
    license_keys = json.load(f)

# Defining license key-value pairs as local variables
locals().update(license_keys)

# Adding license key-value pairs to environment variables
os.environ.update(license_keys)

In [ ]:
# Installing pyspark and spark-nlp
! pip install --upgrade -q pyspark==3.5.1 spark-nlp==$PUBLIC_VERSION

# Installing Spark NLP Healthcare
! pip install --upgrade -q spark-nlp-jsl==$JSL_VERSION  --extra-index-url https://pypi.johnsnowlabs.com/$SECRET

In [3]:
import json
import os

import sparknlp
import sparknlp_jsl

from sparknlp.base import *
from sparknlp.annotator import *
from sparknlp_jsl.annotator import *

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline,PipelineModel

import pandas as pd
pd.set_option('display.max_colwidth', 200)

import warnings
warnings.filterwarnings('ignore')

params = {"spark.driver.memory":"16G",
          "spark.kryoserializer.buffer.max":"2000M",
          "spark.driver.maxResultSize":"2000M"}

print("Spark NLP Version :", sparknlp.version())
print("Spark NLP_JSL Version :", sparknlp_jsl.version())

spark = sparknlp_jsl.start(license_keys['SECRET'],params=params)

spark

Spark NLP Version : 7.0.1
Spark NLP_JSL Version : 7.0.1


## Models

PHI Guard models

| model_name | task | labels | description |
| --- | --- | --- | --- |
| `phi_guard_3m_v0.6` | `leak` | `PHI_LEAK` / `NO_PHI_LEAK` + `phi_leak_prob` | Residual PHI leak in the **final de-identified** note. About 3M parameters, 12.0 MB. |
| `phi_guard_850k_v0.6_fast` | `leak` | `PHI_LEAK` / `NO_PHI_LEAK` + `phi_leak_prob` | Smaller version of `phi_guard_3m_v0.6`: about 850k parameters, 3.5 MB. "Fast" refers to the smaller size, not a measured speed. |
| `phi_guard_3m_v0.6_masked` | `leak` | `PHI_LEAK` / `NO_PHI_LEAK` + `phi_leak_prob` | Residual PHI leak in the **final de-identified** note. About 3M parameters, 12.0 MB. Its model card does not say how it differs from `phi_guard_3m_v0.6`. |
| `phi_guard_3m_v0.6_raw` | `raw` | `PHI_PRESENT` / `NO_PHI_PRESENT` + `phi_present_prob` | PHI presence in the **original** note, before de-identification. About 3M parameters, 12.0 MB. |
| `phi_guard_3m_v0.6_richness` | `richness` | `NONE/TRACE/LOW/MODERATE/HIGH/VERY_HIGH/SATURATED` + `level` | PHI exposure level (ordinal) of the **final de-identified** note. About 3M parameters, 12.0 MB. |
| `phi_guard_3m_v0.7` | `raw`, `leak`, `harmful` or `richness`, <br><br> one per stage, set with `setMode` | `raw`: `PHI_PRESENT` / `NO_PHI_PRESENT` + `phi_present_prob`<br><br>`leak`: `PHI_LEAK` / `NO_PHI_LEAK` + `phi_leak_prob`<br><br>`harmful`: `HARMFUL_PHI_LEAK` / `NO_HARMFUL_PHI_LEAK` + `harmful_leak_prob`<br><br>`richness`: `NONE/TRACE/LOW/MODERATE/HIGH/VERY_HIGH/SATURATED` + `score_1..6`, `level` | Multi-task model: one ONNX graph covers all four tasks. Score `raw` on the original note and the other tasks on the final de-identified note. <br><br>  About 3M parameters, 14.1 MB. `DecisionModel.pretrained()` loads it when no name is given. |
| `phi_guard_870k_v0.7_fast` | `raw`, `leak`, `harmful` or `richness`, <br><br> one per stage, set with `setMode` | `raw`: `PHI_PRESENT` / `NO_PHI_PRESENT` + `phi_present_prob`<br><br>`leak`: `PHI_LEAK` / `NO_PHI_LEAK` + `phi_leak_prob`<br><br>`harmful`: `HARMFUL_PHI_LEAK` / `NO_HARMFUL_PHI_LEAK` + `harmful_leak_prob`<br><br>`richness`: `NONE/TRACE/LOW/MODERATE/HIGH/VERY_HIGH/SATURATED` + `score_1..6`, `level`| Smaller version of `phi_guard_3m_v0.7`: about 870k parameters, 3.6 MB. "Fast" refers to the smaller size, not a measured speed. |

None of the models finds or removes PHI. A negative label does not certify that a note is safe to share.


## DecisionModel Annotator

In [11]:
document_assembler = DocumentAssembler()\
        .setInputCol("text")\
        .setOutputCol("document")

phi_guard_model = DecisionModel.pretrained("phi_guard_3m_v0.6_raw", "en", "clinical/models")\
        .setInputCols("document")\
        .setOutputCol("phi_present")

base_phi_guard_pipeline = Pipeline(
    stages=[
        document_assembler,
        phi_guard_model
    ]
)

phi_guard_3m_v0.6_raw download started this may take some time.
Approximate size to download 11.4 MB


In [14]:
phi_guard_model.getClasses()

['PHI_PRESENT', 'NO_PHI_PRESENT']

In [26]:
notes = [
    ("Example fictional note: Alex Rivera visited Riverside Clinic on 2024-01-15.",),
    ("Example fictional note: <NAME> visited <HOSPITAL> on <DATE>",),
    ("Record date : <DATE> , <NAME> , M.D .",),
    ("Call the clinic at (555) 123-4567 or email <CONTACT> .",),
    ("Paciente Juan Pérez , fecha de ingreso 12/05/2090 .",),
    ("Patient is a <AGE> year old male seen by Dr. <NAME> on <DATE> .",),
    ("Patient's VIN : <VIN>, SSN <LICENSE>, Driver's license <DLN>.",),
    ("Phone (302) 786-5227, 0295 Keats Street, San Francisco.",),
    ("Name : Hendrickson, Ora, Record date: 2093-01-13, Age: 25, # 719435.",),
    ("My name is <NAME> living in <LOCATION>.",)
]

spark_df = spark.createDataFrame(notes, ["text"])
spark_df.show(truncate=100)

+---------------------------------------------------------------------------+
|                                                                       text|
+---------------------------------------------------------------------------+
|Example fictional note: Alex Rivera visited Riverside Clinic on 2024-01-15.|
|                Example fictional note: <NAME> visited <HOSPITAL> on <DATE>|
|                                      Record date : <DATE> , <NAME> , M.D .|
|                     Call the clinic at (555) 123-4567 or email <CONTACT> .|
|                        Paciente Juan Pérez , fecha de ingreso 12/05/2090 .|
|            Patient is a <AGE> year old male seen by Dr. <NAME> on <DATE> .|
|              Patient's VIN : <VIN>, SSN <LICENSE>, Driver's license <DLN>.|
|                    Phone (302) 786-5227, 0295 Keats Street, San Francisco.|
|       Name : Hendrickson, Ora, Record date: 2093-01-13, Age: 25, # 719435.|
|                                    My name is <NAME> living in

In [13]:
result = base_phi_guard_pipeline.fit(spark_df).transform(spark_df)
result.selectExpr("text","phi_present[0].result AS label",
                  "phi_present[0].metadata AS metadata").show(truncate=False)

+---------------------------------------------------------------------------+--------------+--------------------------------------------------------------------------------------------+
|text                                                                       |label         |metadata                                                                                    |
+---------------------------------------------------------------------------+--------------+--------------------------------------------------------------------------------------------+
|Example fictional note: Alex Rivera visited Riverside Clinic on 2024-01-15.|PHI_PRESENT   |{sentence -> 0, status -> scored, n_windows -> 1, phi_present_prob -> 0.9999991401066466}   |
|Example fictional note: <NAME> visited <HOSPITAL> on <DATE>                |NO_PHI_PRESENT|{sentence -> 0, status -> scored, n_windows -> 1, phi_present_prob -> 4.4499577033937134E-5}|
|Record date : <DATE> , <NAME> , M.D .                                

## Choosing Mode

The v7 stage contains four task heads, but one stage emits one selected task. getClasses() fails until setMode(...) is set. raw scores the original text. leak, harmful, and richness score the final de-identified document. Copy the stage for each task instead of changing one stage's mode in place.

- raw ['PHI_PRESENT', 'NO_PHI_PRESENT']
- leak ['PHI_LEAK', 'NO_PHI_LEAK']
- harmful ['HARMFUL_PHI_LEAK', 'NO_HARMFUL_PHI_LEAK']
- richness ['NONE', 'TRACE', 'LOW', 'MODERATE', 'HIGH', 'VERY_HIGH', 'SATURATED']

In [15]:
v7_guard = DecisionModel.pretrained('phi_guard_3m_v0.7', 'en', 'clinical/models')
try:
    print('v7 unset classes:', list(v7_guard.getClasses()))
except Exception as exc:
    print('v7 unset rejected:', type(exc).__name__)
for task in ('raw', 'leak', 'harmful', 'richness'):
    selected = v7_guard.copy().setMode(task)
    print(task, list(selected.getClasses()))

phi_guard_3m_v0.7 download started this may take some time.
Approximate size to download 13.4 MB
v7 unset rejected: IllegalArgumentException
raw ['PHI_PRESENT', 'NO_PHI_PRESENT']
leak ['PHI_LEAK', 'NO_PHI_LEAK']
harmful ['HARMFUL_PHI_LEAK', 'NO_HARMFUL_PHI_LEAK']
richness ['NONE', 'TRACE', 'LOW', 'MODERATE', 'HIGH', 'VERY_HIGH', 'SATURATED']


In [16]:
document_assembler = DocumentAssembler()\
    .setInputCol("text")\
    .setOutputCol("document")

phi_guard_model_richness =  DecisionModel.pretrained('phi_guard_3m_v0.7', 'en', 'clinical/models')\
    .setInputCols("document")\
    .setOutputCol("phi_present")\
    .setMode("richness")

base_phi_guard_pipeline = Pipeline(
    stages=[
        document_assembler,
        phi_guard_model_richness
    ]
)

phi_guard_3m_v0.7 download started this may take some time.
Approximate size to download 13.4 MB


In [17]:
result = base_phi_guard_pipeline.fit(spark_df).transform(spark_df)
result.selectExpr("text","phi_present[0].result AS label",
                  "phi_present[0].metadata AS metadata").show(truncate=False)

+---------------------------------------------------------------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|text                                                                       |label    |metadata                                                                                                                                                                                                                                                               |
+---------------------------------------------------------------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Adding phi guard models to a docwise de-identification pipeline


In [18]:
deid_pipeline = PretrainedPipeline('clinical_deidentification_docwise_benchmark_optimized_v2', 'en', 'clinical/models')

phi_guard_model_leak =  DecisionModel.pretrained('phi_guard_3m_v0.7', 'en', 'clinical/models')\
    .setInputCols("masked")\
    .setOutputCol("phi_present")\
    .setMode("leak")

merged_deid_pipeline = PipelineModel(
    stages=list(deid_pipeline.model.stages) + [phi_guard_model_leak]
)

phi_guard_3m_v0.7 download started this may take some time.
Approximate size to download 13.4 MB


In [23]:
masked_results = merged_deid_pipeline.transform(spark_df)
masked_results.select("masked.result",
                      F.col('phi_present')[0]['result'].alias('label'),
                      F.col('phi_present')[0]['metadata']['status'].alias('status'),
                      F.col('phi_present')[0]['metadata']['phi_leak_prob'].alias('probability')).show(truncate=False)

+-----------------------------------------------------------------+-----------+------+---------------------+
|result                                                           |label      |status|probability          |
+-----------------------------------------------------------------+-----------+------+---------------------+
|[Example fictional note: <NAME> visited <HOSPITAL> on <DATE>.]   |NO_PHI_LEAK|scored|0.0014045593184782064|
|[Example fictional note: <NAME> visited <HOSPITAL> on <DATE>]    |NO_PHI_LEAK|scored|0.001422756461480097 |
|[Record date : <DATE> , <NAME> , M.D .]                          |NO_PHI_LEAK|scored|0.005352475889778607 |
|[Call the clinic at <PHONE> or email <CONTACT> .]                |NO_PHI_LEAK|scored|6.454221474351777E-4 |
|[Paciente <NAME> , fecha de ingreso <DATE> .]                    |NO_PHI_LEAK|scored|0.41876025773167413  |
|[Patient is a <AGE> year old male seen by Dr. <NAME> on <DATE> .]|NO_PHI_LEAK|scored|0.0021780849870124495|
|[Patient's VIN : <

## Filter scored de-identified documents by classification

In [29]:
document_assembler = DocumentAssembler()\
    .setInputCol("text")\
    .setOutputCol("document")

phi_guard_model_leak =  DecisionModel.pretrained('phi_guard_3m_v0.7', 'en', 'clinical/models')\
    .setInputCols("document")\
    .setOutputCol("phi_present")\
    .setMode("leak")
document_filterer = DocumentFiltererByClassifier()\
    .setInputCols(["document", "phi_present"])\
    .setOutputCol("filteredDocuments")\
    .setWhiteList(["NO_PHI_LEAK"])\
    .setCaseSensitive(False)

filter_pipeline = Pipeline().setStages([
    document_assembler,
    phi_guard_model_leak,
    document_filterer
])

phi_guard_3m_v0.7 download started this may take some time.
Approximate size to download 13.4 MB


In [30]:
result = filter_pipeline.fit(spark_df).transform(spark_df)
result.selectExpr("filteredDocuments").show(truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------+
|filteredDocuments                                                                                                                    |
+-------------------------------------------------------------------------------------------------------------------------------------+
|[]                                                                                                                                   |
|[{document, 0, 58, Example fictional note: <NAME> visited <HOSPITAL> on <DATE>, {sentence -> 0, class_label -> NO_PHI_LEAK}, []}]    |
|[{document, 0, 36, Record date : <DATE> , <NAME> , M.D ., {sentence -> 0, class_label -> NO_PHI_LEAK}, []}]                          |
|[]                                                                                                                                   |
|[]                                             

## Medical Decision Models (DMs)

📖 **Reference:** [Medical Decision Models (DMs)](https://nlp.johnsnowlabs.com/docs/en/LLMs/medical_llm#medical-decision-models-dms)

> **For reading only.** The examples in this section and the next two call a running Medical LLM container (`johnsnowlabs/jsl-llms`), so they are not run in this notebook. To run them, follow the reference links.

Medical Decision Models (DMs) are John Snow Labs' compact, task-optimized models that answer typed questions about clinical text: pick one option, answer yes or no with a probability, or choose an ordered level. The question and its options are part of the input, so a new rule or threshold needs no retraining. The PHI Guard models used in this notebook are the CPU members of this family. Medical DM Small and Medical DM Nano are offered for private deployment, separately from the Healthcare NLP library.

| Model | Best for | Runs on | Recommended memory | Max sequence length | Available on |
| --- | --- | --- | --- | --- | --- |
| Medical DM Small | Highest average score across the nine task areas | GPU | ~16 GB GPU VRAM | 16K | On-Premise, AWS, Azure, Databricks, Snowflake |
| Medical DM Nano | Compact; runs on a single small GPU or on CPU | GPU or CPU | ~2 GB GPU VRAM | 16K | On-Premise, AWS, Azure, Databricks, Snowflake |
| PHI Guard (the 7 models above) | PHI leak, harmful leak, richness and raw-note checks | CPU, ONNX | ~1 GB RAM | 8K | Healthcare NLP library, Models Hub |

Across nine clinical task areas (cancer registry abstraction, coding and prior authorization, cohort identification, adverse drug events, general clinical decisions, quality measure review, clinical note review, clinical trial eligibility and oncology record review), Medical DM Small scores **0.917** on average and Medical DM Nano **0.901** (the mean of accuracy and macro precision, recall and F1).

**Example: ask Medical DM Small typed questions (`POST /v1/decisions`)**

Every question has a `type`, plain-language `instructions` and optional `criteria`:

| `type` | Answer | `criteria` |
| --- | --- | --- |
| `noul` | Yes or no: `noul` is the probability of yes | Optional `{"true": "...", "false": "..."}` definitions |
| `choice` | Pick one option: `choice`, `confidence` and `probabilities` | Option key → description |
| `score` | An ordered level: the expected `score` and a probability per level | List of level descriptions, in order |

Start the container with the decision model:

```bash
docker run -d \
    --gpus all \
    --env "SPARK_NLP_LICENSE=your_license_key" \
    -p 8080:8080 \
    --ipc=host \
    johnsnowlabs/jsl-llms:0.33.0 \
    --model Medical-DM-Small \
    --port 8080
```

Send one record (`state`) and up to 64 questions:

```bash
curl -s http://localhost:8080/v1/decisions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "Medical-DM-Small",
    "state": "Discharge summary. Patient Alder Quill was admitted with chest pain. Troponin normal; discharged home on aspirin.",
    "questions": {
      "leak": {
        "type": "noul",
        "instructions": "Is any real identifying detail left in this de-identified note?",
        "criteria": {
          "true": "A real name, phone number, address, date or record number appears in the text",
          "false": "Every identifier is a tag such as [NAME] or [DATE], or has been removed"
        }
      },
      "kind": {
        "type": "choice",
        "instructions": "Which kind of real identifier is left in the note?",
        "criteria": {
          "person_name": "A real person name (a tag like [NAME] does not count)",
          "phone_number": "A real telephone number",
          "none": "No real identifier: only tags such as [NAME] or [DATE]"
        }
      }
    }
  }'
```

Response (illustrative values):

```json
{
  "model": "Medical-DM-Small",
  "answers": {
    "leak": {"type": "noul", "noul": 1.0},
    "kind": {"type": "choice", "choice": "person_name", "confidence": 0.99,
             "probabilities": {"person_name": 0.998, "phone_number": 0.0, "none": 0.002}}
  },
  "usage": {"input_tokens": 142, "output_tokens": 2}
}
```

- `confidence` rescales the top choice probability against chance; send low-confidence answers to a human reviewer.
- The record, one question and its options must fit in 16,384 tokens. Longer input returns `422` instead of being cut short.


## 🛡️ Guard Models (experimental)

📖 **Reference:** [Medical LLM README, section "🛡️ Guard Models (experimental)"](https://github.com/JohnSnowLabs/spark-nlp-workshop/blob/master/products/medllm/README.md)

The same PHI guard idea is also available as an HTTP service in the Medical LLM container. The guards read whole documents and return scores; they do not write or change text. They are small models that run on the CPU next to the decision model `Medical-DM-Small`. They are not chat models and are not listed by `GET /v1/models`.

These are PHI guards v0.7 (experimental, still under test):

| Guard id | Use it for |
| --- | --- |
| `phi-guard-v7-final` | Best accuracy. |
| `phi-guard-v7-fast` | The same outputs, about 5 times faster, a little less accurate. |

Start `Medical-DM-Small` with `--guards` (`all`, or guard ids separated by commas). Without the flag the guard endpoints are not active:

```bash
docker run -d \
    --gpus all \
    --env "SPARK_NLP_LICENSE=your_license_key" \
    -p 8080:8080 \
    --ipc=host \
    johnsnowlabs/jsl-llms:0.33.0 \
    --model Medical-DM-Small \
    --guards all \
    --port 8080
```

**What a guard returns** (every document gets all four; the cutoff for every label is 0.5)

| Output | Fields | Meaning | Use it on |
| --- | --- | --- | --- |
| Leak | `score`, `logit`, `label` | Probability that PHI is still readable in a de-identified note. | De-identified text |
| Harmful leak | `harmful_score`, `harmful_logit`, `harmful_label` | Probability that the leak is harmful (a name, ID, contact detail, date of birth, street address, an age over 89, or three or more harmless kinds together). Never above `score`. | De-identified text |
| Exposure level | `level`, `expected_level`, `level_scores`, `level_logits` | How much PHI a note shows, from 0 (none) to 6 (saturated). `level_scores[k-1]` is the probability that the level is at least `k`. | Either |
| Raw note | `raw_score`, `raw_logit`, `raw_label` | Probability that a raw note, before de-identification, holds PHI at all. | Raw text |

**Example: score documents (`POST /v1/guards`)**

```bash
curl -s http://localhost:8080/v1/guards \
  -H "Content-Type: application/json" \
  -d '{
    "model": "phi-guard-v7-final",
    "docs": [
      {"id": "note-001", "text": "Pt [NAME] seen on [DATE] for chest pain. Troponin negative x2, ECG normal sinus rhythm. Discharged home on aspirin 81 mg daily."},
      {"id": "note-002", "text": "Pt Maria Delgado, DOB 03/14/1956, MRN 4471923, seen on [DATE] for chest pain. Discharged home on aspirin 81 mg daily."}
    ]
  }'
```

Response (the second result is shown in full):

```json
{
  "model": "phi-guard-v7-final",
  "default_cutoff": 0.5,
  "results": [
    {"id": "note-001", "score": 0.0101, "logit": -4.5816, "label": false, "cutoffs": {"0.5": false}, "...": "..."},
    {"id": "note-002", "score": 1.0, "logit": 15.9447, "label": true, "cutoffs": {"0.5": true},
     "harmful_score": 1.0, "harmful_logit": 11.105, "harmful_label": true,
     "level": 3, "expected_level": 3.4924,
     "level_scores": [1.0, 1.0, 1.0, 0.4923, 0.0001, 0.0],
     "level_logits": [23.6157, 20.1183, 12.1113, -0.0306, -9.1865, -15.1122],
     "raw_score": 1.0, "raw_logit": 31.8373, "raw_label": true}
  ],
  "usage": {"docs": 2, "windows": 2, "ms": 12}
}
```

- Send up to 16,384 documents and 50 MiB per request. Every document needs a unique `id` and a string `text`; results come back in the order sent.
- Long documents are read in 1,024-byte windows. Very confident scores all round to 1.0, so rank documents with the `logit` fields.
- Send masked or raw documents. Do not send an obfuscated copy (real PHI replaced by realistic fake names, dates and numbers): it looks like real PHI to the guard and raises false flags.
- `GET /v1/guards` lists the guards loaded in the container, and `http://localhost:8080/openapi-doc` serves an interactive API reference.


## Medical-DM-Small and PHI Guards

📖 **Reference notebook:** [Medical_DM_Small_and_PHI_Guards.ipynb](https://github.com/JohnSnowLabs/spark-nlp-workshop/blob/master/products/medllm/Medical_DM_Small_and_PHI_Guards.ipynb)

The reference notebook calls every endpoint of a `Medical-DM-Small` container started with the PHI guards from Python, and shows what you send, what comes back and how to read it. All notes in it are invented.

| Endpoint | What it does |
| --- | --- |
| `GET /health`, `GET /v1/models` | Is the server ready, and which decision model it serves. |
| `GET /v1/guards` | The guards loaded in the container. |
| `POST /v1/guards` (experimental) | Checks documents with one guard: is PHI left in a de-identified note, is it harmful, how much PHI, does a raw note hold PHI. Runs on the CPU. |
| `POST /v1/guards/ensemble` (experimental) | Checks documents with a guard **and** with `Medical-DM-Small`, combines the two scores, and routes each document to `leak`, `review` or `clean`. |
| `POST /v1/decisions` | Asks the decision model typed questions (yes/no, choice, score) about a text. |

**Example: one guard from Python**

```python
import requests

BASE_URL = "http://localhost:8080"

def call(method, path, payload=None):
    """Send a request to the container and return (HTTP status, JSON answer)."""
    response = requests.request(method, BASE_URL + path, json=payload, timeout=300)
    return response.status_code, response.json()

CLEAN = "Pt [NAME] seen on [DATE] for chest pain. Troponin negative x2, ECG normal sinus rhythm. Discharged home on aspirin 81 mg daily."
LEAK = "Pt Maria Delgado, DOB 03/14/1956, MRN 4471923, seen on [DATE] for chest pain. Discharged home on aspirin 81 mg daily."

_, answer = call("POST", "/v1/guards", {
    "model": "phi-guard-v7-final",
    "docs": [{"id": "note-001", "text": CLEAN}, {"id": "note-002", "text": LEAK}],
})
```

Result in the reference notebook, one row per document:

| id | score | label | harmful_score | harmful_label | level | level_name | raw_score |
| --- | --- | --- | --- | --- | --- | --- | --- |
| note-001 | 0.0101 | False | 0.0028 | False | 0 | none | 0.0 |
| note-002 | 1.0000 | True | 1.0000 | True | 3 | moderate | 1.0 |

`note-001` has every identifier masked, so its leak `score` is close to 0. `note-002` still shows a name, a date of birth and a record number, so it is a harmful leak with a level above 0.

**Example: a guard and the decision model together (`POST /v1/guards/ensemble`)**

The ensemble checks each document with a guard **and** with `Medical-DM-Small`, combines the two scores (by default `mean_logit`), and routes the document:

| `decision` | Meaning |
| --- | --- |
| `leak` | Both models flag the document. |
| `review` | Only one model flags it: send it to a person. |
| `clean` | Neither flags it. |

```python
_, answer = call("POST", "/v1/guards/ensemble", {
    "model": "phi-guard-ensemble",   # optional fields: "guard", "aggregation", "decision"
    "docs": [{"id": "note-001", "text": CLEAN}, {"id": "note-002", "text": LEAK}],
})

for r in answer["results"]:
    print(r["id"], r["decision"], r["score"], r["members"])
```

The reference notebook screens eight notes this way, 64 documents per request. Only the note the two models disagree on goes to review:

| note | route | combined | guard | decision model |
| --- | --- | --- | --- | --- |
| masked | clean | 0.0010 | 0.0101 | 0.0000 |
| tags only | clean | 0.0003 | 0.0009 | 0.0000 |
| no PHI | clean | 0.0003 | 0.0009 | 0.0000 |
| name, date of birth and record number left | leak | 1.0000 | 1.0000 | 1.0000 |
| phone number left | leak | 0.9998 | 0.9999 | 0.9994 |
| e-mail address left | leak | 0.9995 | 0.9978 | 1.0000 |
| member id left | leak | 0.9999 | 1.0000 | 1.0000 |
| 93-year-old patient | review | 0.7206 | 0.9986 | 0.0090 |

The reference notebook also shows long notes, raw notes, the four aggregations (`mean_logit`, `mean`, `min`, `max`), bringing your own `decision_score`, and error codes. It also shows how to get a masked note with the Healthcare NLP pipeline `clinical_deidentification_docwise_wip` before sending it to the guards; send the masked output, not the obfuscated one.
